In [33]:
from google.colab import drive
import torch
import os
import glob
import numpy as np
import cv2

In [34]:
#Mount Google Drive and navigate to the assignment directory
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignments/Video_action_recognition
!git pull
%pip install -q -r requirements.txt
%env PYTHONPATH=src
# Check for PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/assignments/Video_action_recognition
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 977 bytes | 20.00 KiB/s, done.
From https://github.com/ttl074/Video_action_recognition
   536676e..e1c6433  main       -> origin/main
Updating 536676e..e1c6433
Fast-forward
 src/utils.py | 64 +++++++++++++++++++++++++++++++++++++++++++-----------------
 1 file changed, 46 insertions(+), 18 deletions(-)
env: PYTHONPATH=src
PyTorch version: 2.11.0+cu130
CUDA available: True


In [26]:
# Setup paths for dataset
drive_path = '/content/drive/MyDrive/datasets/ucf50'
local_path = '/content'
url = 'https://www.crcv.ucf.edu/data/UCF50/UCF50.rar'
extract_path = os.path.join(local_path, 'datasets')
raw_dir = os.path.join(extract_path, 'UCF50')
frame_dir = os.path.join(extract_path,'UCF50_frames_16')

In [27]:
!python scripts/prep_dataset.py --drive_path {drive_path} --local_path {local_path} --url {url}

Local rar file already exists. Skipping download.
Extract path already exists at /content/datasets/UCF50. Skipping extraction.


In [18]:
video_counts = {}
frame_counts = []
failed_videos = []
for category in os.listdir('/content/datasets/UCF50'):
    category_path = os.path.join('/content/datasets/UCF50', category)
    video_count = 0
    for video in os.listdir(category_path):
        video_path = os.path.join(category_path, video)
        v_cap = cv2.VideoCapture(video_path)
        if not v_cap.isOpened():
            print("Failed to open video:", video_path)
            failed_videos.append(video_path)
            v_cap.release()
            continue
        v_len = int(v_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_cap.release()
        frame_counts.append(v_len)
        video_count += 1
    video_counts[category] = video_count
largest_category = max(video_counts, key=video_counts.get)
smallest_category = min(video_counts, key=video_counts.get)
print("Number of videos:", sum(video_counts.values()))
print("Largest category:", largest_category, video_counts[largest_category])
print("Smallest category:", smallest_category, video_counts[smallest_category])
print("Median frame count:", np.median(frame_counts))
print("Min frame count:", np.min(frame_counts))
print("Max frame count:", np.max(frame_counts))
print("Failed videos:", failed_videos)

Number of videos: 6681
Largest category: HorseRiding 197
Smallest category: TaiChi 100
Median frame count: 181.0
Min frame count: 29
Max frame count: 900
Failed videos: []


In [35]:
n_frames = 16
!python scripts/extract_frames.py --raw_dir {raw_dir} --n_frames {n_frames} --frame_dir {frame_dir}

Converting from avi: 100% 6681/6681 [04:47<00:00, 23.25file/s]
Extracted 6662 and skipped 19 out of 6681 avis


In [36]:
!python -m pytest -v -rs tests/

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/assignments/Video_action_recognition
configfile: pytest.ini
plugins: platformdirs-4.13.3, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.11
collected 6 items                                                              

tests/test_dataset.py::test_raw_class_folders PASSED                     [ 16%]
tests/test_dataset.py::test_video_avis PASSED                            [ 33%]
tests/test_dataset.py::test_ucf50_frames_dir PASSED                      [ 50%]
tests/test_dataset.py::test_video_dirs PASSED                            [ 66%]
tests/test_dataset.py::test_frame_counts PASSED                          [ 83%]
tests/test_dataset.py::test_frames_read PASSED                           [100%]

============================== 6 passed in 5.01s ===============================
